In [1]:
from dataclasses import dataclass
from typing import Optional,Tuple,List
from collections import namedtuple

import math
import time
import json
import torch
import torch.nn as nn
import numpy as np
from torch import Tensor
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint

from transformers.modeling_utils import PreTrainedModel


def log(t, eps = 1e-20):
    return torch.log(t.clamp(min = eps))


def gumbel_noise(t):
    noise = torch.zeros_like(t).uniform_(0, 1)
    return -log(-log(noise))


def _topk(array, k:int):
    top_k_indices = torch.topk(array, k)[-1]
    one_hot_length = array.shape[-1]
    one_hot_indices = torch.nn.functional.one_hot(top_k_indices, one_hot_length).to(array.dtype)
    top_k_values = torch.einsum('...s,...is->...i', array, one_hot_indices)
    return top_k_values, top_k_indices, one_hot_indices.max(2)[0]


def one_hot_with_ignore(indices, num_classes, dtype=torch.int32):
    shape = indices.shape
    one_hot = torch.zeros(indices.shape + (num_classes,), dtype=dtype, device=indices.device)
    flattened_indices = indices.reshape(-1)
    flattened_one_hot = one_hot.reshape(-1, num_classes)
    mask = (flattened_indices >= 0) & (flattened_indices < num_classes)
    flattened_one_hot[mask, flattened_indices[mask]] = 1
    flattened_one_hot[~mask] = 0
    return one_hot

    
def _load_balancing_loss(router_probs, expert_indices, mask=None) -> float:
  num_experts = router_probs.shape[-1]
  # Shape: [num_groups, tokens_per_group, num_selected_experts, num_experts].
  expert_mask = torch.nn.functional.one_hot(expert_indices, num_experts)
  # Shape: [num_groups, tokens_per_group, num_experts]
  expert_mask = expert_mask.max(2)[0].to(torch.float32)
  if mask is None:
      tokens_per_group_and_expert = expert_mask.mean(1)
      router_prob_per_group_and_expert = router_probs.mean(1)
  else:
      tokens_per_group_and_expert = expert_mask.sum(1) / mask.sum(1)
      router_prob_per_group_and_expert = router_probs.sum(1) / mask.sum(1)
  p = tokens_per_group_and_expert * router_prob_per_group_and_expert
  return p.mean() * num_experts**2

In [2]:
class MoeBlock(nn.Module):
    def __init__(self, config) -> None:
        super(MoeBlock, self).__init__()
        self.config = config
        self.min_group_size = 1
        self.num_experts = getattr(config, 'num_experts', 8)
        self.dim = getattr(config, 'dim', 4096)
        self.intermediate_size = getattr(config, 'intermediate_size', 5632)
        self.mgate_dim = getattr(config, 'mgate_dim', 44)
        self.topn = getattr(config, 'num_experts_per_tok', 2)
        self.expert_capacity_factor = getattr(config, 'expert_capacity_factor', 1.5)
        self.gate_noise_coef = getattr(config, 'gate_noise_coef', 0.5)
        self.sfm_after_topn = getattr(config, 'sfm_after_topn', True)
        self.dtype = getattr(config, 'torch_dtype', torch.bfloat16)
        self.aux_loss_coef = getattr(config, 'aux_loss_coef', 0.01)
        self.router_z_loss_coef = getattr(config, 'router_z_loss_coef', 0.01)
        self.expert_chunk_size = getattr(config, 'expert_chunk_size', None)
        self.mgate = getattr(config, 'mgate', False)

        self.wi_gate_0 = nn.Parameter(torch.empty(self.num_experts, self.dim, self.intermediate_size))
        self.wi_0 = nn.Parameter(torch.empty(self.num_experts, self.dim, self.intermediate_size))
        self.wo_0 = nn.Parameter(torch.empty(self.num_experts, self.intermediate_size, self.dim))
        
        # nn.init.normal_(self.wi_gate_0, mean=0, std=0.006)
        # nn.init.normal_(self.wi_0, mean=0, std=0.006)
        # nn.init.normal_(self.wo_0, mean=0, std=0.006)
        nn.init.constant_(self.wi_gate_0, 1.0)
        nn.init.constant_(self.wi_0, 1.0)
        nn.init.constant_(self.wo_0, 1.0)

        self.router_gate = nn.Parameter(torch.empty(self.dim, self.num_experts))
        # nn.init.normal_(self.router_gate, mean=0, std=0.006)
        nn.init.constant_(self.router_gate, 1.0)
        
        if self.mgate:
            self.mg = nn.Parameter(torch.empty(self.num_experts, self.dim, self.mgate_dim))
            # nn.init.normal_(self.mg, mean=0, std=0.006)
            nn.init.constant_(self.mg, 1.0)
            
    def _call_experts(self, expert_inputs, expert_index, compute_n_expert, training=False):
        theta_wi = self.wi_0[expert_index: expert_index + compute_n_expert]
        theta_wo = self.wo_0[expert_index: expert_index + compute_n_expert]
        theta_wi_gated = self.wi_gate_0[expert_index: expert_index + compute_n_expert]
        hidden0 = torch.einsum("gecm,emh->gech", expert_inputs, theta_wi)
        hidden1 = torch.einsum("gecm,emh->gech", expert_inputs, theta_wi_gated)
        hidden1 = torch.nn.functional.silu(hidden1)
        hidden = hidden1 * hidden0
        # expert_inputs: gecm,  mgatew: meh  -> 
        if self.mgate:
          assert isinstance(self.mgate_dim, int)
          inner_gate = self.mg[expert_index: expert_index + compute_n_expert]
          mgate_scores = torch.einsum('gecm,emi->geci', expert_inputs, inner_gate)
          mgate_scores = torch.nn.functional.softmax(mgate_scores.to(torch.float32), dim=-1)
          mgate_scores = mgate_scores.to(self.dtype)
          G, E, C, H = hidden.shape
          hidden = hidden.reshape(G, E, C, self.mgate_dim, H // self.mgate_dim)
          hidden = torch.einsum('geci,gecif->gecif', mgate_scores, hidden)
          hidden = hidden.reshape(G, E, C, H)
        hidden = torch.einsum("gech,ehm->gecm", hidden, theta_wo)
        return hidden

    def forward(self, inputs, paddings=None):
        num_groups = inputs.shape[0]
        num_tokens = inputs.shape[0]*inputs.shape[1]
        # num_tokens = np.prod(inputs.shape[:-1]).to(inputs.device)
        tokens_per_group = num_tokens // num_groups
        assert num_tokens % num_groups == 0, print(f'‘num_tokens % num_groups -> {num_tokens} % {num_groups} != 0’')
        # print(f'expert_capacity_factor: {self.expert_capacity_factor}')
        
        expert_capacity = math.ceil(self.expert_capacity_factor * tokens_per_group / self.num_experts)
        max_group_size = int(inputs.shape[1])
        expert_capacity = min(expert_capacity, max_group_size)
        expert_capacity = max(expert_capacity, self.min_group_size)
        # print(f'expert_capacity: {expert_capacity}')

        grouped_inputs = torch.reshape(inputs, (num_groups, tokens_per_group, self.dim))
        router_logits = torch.einsum('gsm,me->gse', grouped_inputs.to(torch.float32), self.router_gate.to(torch.float32))
        __import__('ipdb').set_trace()

        if self.gate_noise_coef > 0.0:
        #   print(f'gate_noise_coef: {self.gate_noise_coef}')
          noise = gumbel_noise(router_logits)
          router_logits += noise * self.gate_noise_coef
        # one_hot_indices: b l e  expert_index: b l topn
        _, expert_index, one_hot_indices = _topk(router_logits, k=self.topn)
        
        if self.sfm_after_topn:
          assert one_hot_indices is not None
          router_mask = (1 - one_hot_indices) * torch.finfo(self.dtype).min
          _router_logits = router_logits + router_mask
          router_probs = torch.nn.functional.softmax(_router_logits.to(torch.float32), dim=-1)
        else:
            # gse
          router_probs = torch.nn.functional.softmax(router_logits.to(torch.float32), dim=-1)
            
        router_probs = router_probs.to(self.dtype) # ble
        if paddings is not None:
            # the one means reserved in paddings
            gate_mask = torch.reshape(paddings, grouped_inputs.shape[:2])
            gate_mask = gate_mask.unsqueeze(-1) # bl1
            router_probs *= gate_mask # ble
        else:
            gate_mask = None
        
        aux_loss, router_z_loss = 0.0, 0.0
        if self.aux_loss_coef is not None:
            aux_loss = _load_balancing_loss(router_probs, expert_index, gate_mask)
            aux_loss *= self.aux_loss_coef
            # print(f'aux_loss: {aux_loss}')

        if self.router_z_loss_coef is not None: 
             # The purpose is to prevent the output of the router from becoming too extreme or unstable, to ensure that 
             # the probability distribution is not concentrated on a very small number of experts, and to prevent excessively large logits.
            # <=> torch.logsumexp(logits, dim = -1)
            router_z_loss = torch.logsumexp(router_logits, dim = -1)
            router_z_loss = router_z_loss.square()            
            router_z_loss = self.router_z_loss_coef * router_z_loss.mean()
            # print(f'router_z_loss: {router_z_loss}')

        if paddings is not None:
            expert_index *= (2 * gate_mask - 1) # lsp:masked expert set to negative, it would not bd considered when use function `one_hot_with_ignore`
            no_gate_mask = gate_mask - 1
            expert_index += no_gate_mask.repeat(1, 1, expert_index.shape[-1])
            
        aux_loss = aux_loss + router_z_loss
        # g * 2 * s
        expert_index = expert_index.permute(0, 2, 1)
        # g * 2s
        expert_index = expert_index.reshape(num_groups, -1)
        # g * 2s * e, expert_index , this function can ignore negative
        expert_mask = one_hot_with_ignore(expert_index, self.num_experts, dtype=torch.int32)
        # # g * 2s * e 
        token_priority = torch.cumsum(expert_mask, dim=1) * expert_mask - 1.0
        # # g * 2 * s * e
        token_priority = token_priority.reshape(num_groups, self.topn, -1, self.num_experts)
        # # g * s * 2 * e  lsp: per token select 2 expert，expert corresponss to position value mean current rank expert selected token numbers
        token_priority = token_priority.permute(0, 2, 1, 3)
        token_priority = token_priority.max(2)[0].to(torch.int32) # (b*l) * e
        if self.expert_chunk_size is None:
            compute_n_expert = self.num_experts
        else:
            compute_n_expert = self.num_experts // self.expert_chunk_size
            assert self.num_experts % self.expert_chunk_size == 0, print(self.num_experts, self.expert_chunk_size)
        combined_outputs = None
        print(f'compute_n_expert: {compute_n_expert}')
        for expert_index in range(0, token_priority.shape[-1], compute_n_expert):
            _token_priority = token_priority[..., expert_index: expert_index+compute_n_expert]
            _router_probs = router_probs[..., expert_index: expert_index+compute_n_expert].to(self.dtype)
            # lsp： _dispatch_mask: (g*s)ec
            _dispatch_mask = one_hot_with_ignore(_token_priority.reshape(-1, _token_priority.shape[-1]), expert_capacity, dtype=torch.int32)
            _dispatch_mask = _dispatch_mask.reshape(num_groups, tokens_per_group, compute_n_expert, -1).to(self.dtype).to(_router_probs.device)
            _combine_array = torch.einsum('gse,gsec->gsec', _router_probs, _dispatch_mask)
            _combine_array = _combine_array.to(self.dtype)
            # expert inputs mask：gsm x gsec -> gecm，  _dispatch_mask can drop unused token
            print(grouped_inputs.dtype, _dispatch_mask.dtype)
            _expert_inputs = torch.einsum('gsd,gsec->gecd', grouped_inputs, _dispatch_mask)
            # g * e * c * m
            _expert_outputs = self._call_experts(_expert_inputs, expert_index, compute_n_expert, training=self.training)
            _combined_outputs = torch.einsum('gecm,gsec->gsm', _expert_outputs, _combine_array)
            combined_outputs = _combined_outputs if combined_outputs is None else combined_outputs + _combined_outputs
        combined_outputs = combined_outputs.reshape(*inputs.shape)
        return combined_outputs


In [3]:
class config:
    dim = 128
    base_emb_dim = 128
    num_experts_per_tok = 2
    expert_capacity_factor = 1.5
    min_group_size = 1
    router_z_loss_coef = 0.01
    aux_loss_coef = 0.01
    expert_chunk_size = 1
    mlp_activations = ['silu', 'linear']
    mgate = True
    mgate_dim = 44
    sfm_after_topn = True
    base_mlp_dim = 1408
    gate_noise_coef = 0.0
    init_weights_seed = 9876
    record_internal_nn_metrics = 0
    intermediate_size = 1408
import pickle

inputs = torch.from_numpy(pickle.load(open('inputs.pkl', 'rb'))).to(torch.bfloat16)
model = MoeBlock(config=config)
router_gate = torch.from_numpy(pickle.load(open('router_gate.pkl', 'rb')))
model.router_gate.data = router_gate
model.to(torch.bfloat16)

MoeBlock()

In [4]:
with torch.no_grad():
    outputs = model(inputs)

> /tmp/ipykernel_26561/925109195.py(80)forward()
     79 
---> 80         if self.gate_noise_coef > 0.0:
     81         #   print(f'gate_noise_coef: {self.gate_noise_coef}')



ipdb>  self.router_gate


Parameter containing:
tensor([[ 9.0820e-02,  7.7148e-02, -1.0559e-02,  ...,  1.2158e-01,
         -1.4160e-01,  7.1777e-02],
        [-1.1377e-01,  1.8188e-02, -1.5918e-01,  ..., -2.7161e-03,
         -6.5430e-02,  8.8867e-02],
        [ 2.3499e-03,  1.9165e-02,  3.0060e-03,  ..., -1.4591e-04,
          7.9102e-02, -2.3242e-01],
        ...,
        [ 8.7402e-02,  1.8848e-01, -4.5898e-02,  ..., -2.3804e-03,
          4.0894e-03,  7.1289e-02],
        [ 6.0791e-02,  7.4219e-02, -8.9355e-02,  ..., -2.1973e-01,
         -5.8105e-02, -8.3496e-02],
        [-3.5400e-02, -1.3867e-01,  1.8262e-01,  ...,  1.1670e-01,
          3.5156e-02,  8.9844e-02]], dtype=torch.bfloat16, requires_grad=True)


ipdb>  router_logits


tensor([[[ 0.8118,  1.2509, -0.0583, -0.4191,  2.3464, -0.4159,  0.2824,
          -0.0898],
         [-3.4987, -0.6682, -0.1952,  1.9302,  1.5394,  1.3372, -0.3169,
          -1.2520],
         [ 0.1544,  0.8262, -0.8099,  0.7639, -0.4948,  0.5542,  0.4728,
           1.3659],
         [ 1.3172, -0.1364, -1.0021, -1.4347,  1.2703,  0.4082, -0.7995,
          -0.8045],
         [-1.0446, -1.0110, -0.9342,  1.3721,  0.7014,  0.1559, -0.3271,
          -0.1414],
         [-0.2759,  1.1137, -0.9509, -1.4407, -1.8391, -1.8188,  0.1610,
           2.0125],
         [ 1.1879,  0.5755, -0.4101, -2.6228,  0.0767,  0.8379, -1.0107,
           0.0162],
         [ 0.2709,  0.2616, -0.6260,  1.8525,  0.2547, -1.2250,  0.1738,
          -0.1694],
         [ 0.9204,  0.1619, -0.7450, -0.0883,  1.2816,  0.5958, -1.9378,
           0.1129],
         [-0.2103,  1.0133, -0.3927, -1.2793, -1.3566,  0.2988, -0.9836,
          -0.5555]]])


...

ipdb>  n


> /tmp/ipykernel_26561/925109195.py(161)forward()
    160             combined_outputs = _combined_outputs if combined_outputs is None else combined_outputs + _combined_outputs
--> 161         combined_outputs = combined_outputs.reshape(*inputs.shape)
    162         return combined_outputs



ipdb>  combined_outputs


tensor([[[4.7607e-02, 4.7607e-02, 4.7607e-02,  ..., 4.7607e-02,
          4.7607e-02, 4.7607e-02],
         [3.4880e+03, 3.4880e+03, 3.4880e+03,  ..., 3.4880e+03,
          3.4880e+03, 3.4880e+03],
         [1.9760e+03, 1.9760e+03, 1.9760e+03,  ..., 1.9760e+03,
          1.9760e+03, 1.9760e+03],
         ...,
         [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00],
         [8.9000e+01, 8.9000e+01, 8.9000e+01,  ..., 8.9000e+01,
          8.9000e+01, 8.9000e+01],
         [2.5920e+03, 2.5920e+03, 2.5920e+03,  ..., 2.5920e+03,
          2.5920e+03, 2.5920e+03]]], dtype=torch.bfloat16)


ipdb>  c


In [7]:
for k, v in model.named_parameters():
    print(k, v.shape, v.sum().item(), v.mean().item())

wi_gate_0 torch.Size([8, 128, 1408]) 1441792.0 1.0
wi_0 torch.Size([8, 128, 1408]) 1441792.0 1.0
wo_0 torch.Size([8, 1408, 128]) 1441792.0 1.0
router_gate torch.Size([128, 8]) -3.546875 -0.0034637451171875
mg torch.Size([8, 128, 44]) 45056.0 1.0


In [6]:
outputs

tensor([[[4.7607e-02, 4.7607e-02, 4.7607e-02,  ..., 4.7607e-02,
          4.7607e-02, 4.7607e-02],
         [3.4880e+03, 3.4880e+03, 3.4880e+03,  ..., 3.4880e+03,
          3.4880e+03, 3.4880e+03],
         [1.9760e+03, 1.9760e+03, 1.9760e+03,  ..., 1.9760e+03,
          1.9760e+03, 1.9760e+03],
         ...,
         [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00],
         [8.9000e+01, 8.9000e+01, 8.9000e+01,  ..., 8.9000e+01,
          8.9000e+01, 8.9000e+01],
         [2.5920e+03, 2.5920e+03, 2.5920e+03,  ..., 2.5920e+03,
          2.5920e+03, 2.5920e+03]]], dtype=torch.bfloat16)